In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
import gc
import io
import os
from itertools import combinations

from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

pd.reset_option('display.float_format')
pd.set_option('display.max_colwidth', None)

from config import ROOT, prev_num_aggregations  # lib này được khởi tạo ban đầu dự án

import helpers.view as view
import helpers.EDA as EDA
import modules.utils as utils

importlib.reload(view)
importlib.reload(EDA)
importlib.reload(utils)

from helpers.cache_clear import cache_clear

get_pickle = utils.get_pickle
get_pickles = utils.get_pickles

In [2]:
KEY = "SK_ID_CURR"

In [3]:
col_binary = ['NAME_CONTRACT_TYPE', 'NAME_CONTRACT_STATUS', 'CODE_REJECT_REASON',
              'NAME_YIELD_GROUP', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 
              'NAME_PRODUCT_TYPE', 'NAME_SELLER_INDUSTRY', 'CHANNEL_TYPE',
              'NAME_PAYMENT_TYPE']

In [4]:
prev = utils.get_pickles("prev", cols=[KEY, "DAYS_DECISION"]+col_binary)

In [5]:
prev.sort_values(['SK_ID_CURR', 'DAYS_DECISION'], inplace=True) # top latest

In [6]:
prev.head(10)

,SK_ID_CURR,DAYS_DECISION,NAME_CONTRACT_TYPE,NAME_CONTRACT_STATUS,CODE_REJECT_REASON,NAME_YIELD_GROUP,NAME_GOODS_CATEGORY,NAME_PORTFOLIO,NAME_PRODUCT_TYPE,NAME_SELLER_INDUSTRY,CHANNEL_TYPE,NAME_PAYMENT_TYPE
0,100001,-1740,Consumer loans,Approved,XAP,high,Mobile,POS,XNA,Connectivity,Country-wide,Cash through the bank
1,100002,-606,Consumer loans,Approved,XAP,low_normal,Vehicles,POS,XNA,Auto technology,Stone,XNA
2,100003,-2341,Consumer loans,Approved,XAP,middle,Consumer Electronics,POS,XNA,Consumer electronics,Country-wide,Cash through the bank
3,100003,-828,Consumer loans,Approved,XAP,middle,Furniture,POS,XNA,Furniture,Stone,Cash through the bank
4,100003,-746,Cash loans,Approved,XAP,low_normal,XNA,Cash,x-sell,XNA,Credit and cash offices,XNA
5,100004,-815,Consumer loans,Approved,XAP,middle,Mobile,POS,XNA,Connectivity,Regional / Local,Cash through the bank
6,100005,-757,Consumer loans,Approved,XAP,high,Mobile,POS,XNA,Connectivity,Country-wide,Cash through the bank
7,100005,-315,Cash loans,Canceled,XAP,XNA,XNA,XNA,XNA,XNA,Credit and cash offices,XNA
8,100006,-617,Consumer loans,Approved,XAP,middle,Construction Materials,POS,XNA,Construction,Stone,XNA
9,100006,-438,Cash loans,Approved,XAP,high,XNA,Cash,x-sell,XNA,Credit and cash offices,Cash through the bank


# multiprocess

In [7]:
col_binary_di = {}

for c in col_binary:
    col_binary_di[c] = list(prev[c].unique())

In [8]:
import logging

logging.basicConfig(level=logging.INFO)

In [9]:
def to_decimal(x):
    if len(x) == 0:
        return -1
    return float(str(x[0]) + '.' + ''.join(map(str, x[1:])))

def multi(df):
    is_app = (df['NAME_CONTRACT_STATUS'] == 'Approved')
    is_ref = (df['NAME_CONTRACT_STATUS'] == 'Refused')
    is_appref = is_app | is_ref

    di = {}
    for c in col_binary:
        for v in col_binary_di[c]:
            arr = (df[c] == v).astype(int).values
            
            arr_app = arr[is_app.values]
            arr_ref = arr[is_ref.values]
            arr_appref = arr[is_appref.values]
            
            di[f'{c}-{v}'] = to_decimal(arr)
            di[f'{c}-{v}_app'] = to_decimal(arr_app)
            di[f'{c}-{v}_ref'] = to_decimal(arr_ref)
            di[f'{c}-{v}_appref'] = to_decimal(arr_appref)
            # logging.info(f"{v}, {arr}")
            print(v, arr, flush=True)

            
        break

    return pd.Series(di)

In [10]:
from multiprocessing import Pool, cpu_count
NTHREAD = cpu_count() - 1

In [11]:
grouped = prev.groupby(KEY)
ids = [group for _, group in grouped]

In [12]:
pool = Pool(NTHREAD)

In [13]:
callback = pool.map(multi, ids)

In [ ]:
pool.close()
pool.join()

In [ ]:
base = pd.concat(callback, axis=0)

In [ ]:
base.reset_index(inplace=True)

# dùng 106_decimal.py để chạy multi processing, chạy trên jupyter lỗi quá, các tiến trình không được đưa vào luồng hệ thống 😾

In [6]:
base = pd.read_pickle(ROOT + "/data/feature/base.p").drop(['index'], axis=1)

In [7]:
_base = pd.DataFrame()
_base[KEY] = prev[KEY].unique()

In [8]:
base = pd.concat([_base, base], axis=1)

In [9]:
base

,SK_ID_CURR,NAME_CONTRACT_STATUS-Approved,NAME_CONTRACT_STATUS-Approved_app,NAME_CONTRACT_STATUS-Approved_ref,NAME_CONTRACT_STATUS-Approved_appref,NAME_CONTRACT_STATUS-Canceled,NAME_CONTRACT_STATUS-Canceled_app,NAME_CONTRACT_STATUS-Canceled_ref,NAME_CONTRACT_STATUS-Canceled_appref,NAME_CONTRACT_STATUS-Refused,NAME_CONTRACT_STATUS-Refused_app,NAME_CONTRACT_STATUS-Refused_ref,NAME_CONTRACT_STATUS-Refused_appref,NAME_CONTRACT_STATUS-Unused offer,NAME_CONTRACT_STATUS-Unused offer_app,NAME_CONTRACT_STATUS-Unused offer_ref,NAME_CONTRACT_STATUS-Unused offer_appref,CODE_REJECT_REASON-XAP,CODE_REJECT_REASON-XAP_app,CODE_REJECT_REASON-XAP_ref,CODE_REJECT_REASON-XAP_appref,CODE_REJECT_REASON-LIMIT,CODE_REJECT_REASON-LIMIT_app,CODE_REJECT_REASON-LIMIT_ref,CODE_REJECT_REASON-LIMIT_appref,CODE_REJECT_REASON-HC,CODE_REJECT_REASON-HC_app,CODE_REJECT_REASON-HC_ref,CODE_REJECT_REASON-HC_appref,CODE_REJECT_REASON-CLIENT,CODE_REJECT_REASON-CLIENT_app,CODE_REJECT_REASON-CLIENT_ref,CODE_REJECT_REASON-CLIENT_appref,CODE_REJECT_REASON-SCO,CODE_REJECT_REASON-SCO_app,CODE_REJECT_REASON-SCO_ref,CODE_REJECT_REASON-SCO_appref,CODE_REJECT_REASON-SCOFR,CODE_REJECT_REASON-SCOFR_app,CODE_REJECT_REASON-SCOFR_ref,CODE_REJECT_REASON-SCOFR_appref,CODE_REJECT_REASON-VERIF,CODE_REJECT_REASON-VERIF_app,CODE_REJECT_REASON-VERIF_ref,CODE_REJECT_REASON-VERIF_appref,CODE_REJECT_REASON-XNA,CODE_REJECT_REASON-XNA_app,CODE_REJECT_REASON-XNA_ref,CODE_REJECT_REASON-XNA_appref,CODE_REJECT_REASON-SYSTEM,CODE_REJECT_REASON-SYSTEM_app,CODE_REJECT_REASON-SYSTEM_ref,CODE_REJECT_REASON-SYSTEM_appref
0,100001,1.00000,1.00000,-1.0,1.00000,0.0,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,1.00000,1.00000,-1.0,1.00000,0.0,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0
1,100002,1.00000,1.00000,-1.0,1.00000,0.0,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,1.00000,1.00000,-1.0,1.00000,0.0,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0
2,100003,1.11000,1.11000,-1.0,1.11000,0.0,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,1.11000,1.11000,-1.0,1.11000,0.0,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0
3,100004,1.00000,1.00000,-1.0,1.00000,0.0,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,1.00000,1.00000,-1.0,1.00000,0.0,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0
4,100005,1.00000,1.00000,-1.0,1.00000,0.1,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,1.10000,1.00000,-1.0,1.00000,0.0,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
338852,456251,1.00000,1.00000,-1.0,1.00000,0.0,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,1.00000,1.00000,-1.0,1.00000,0.0,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0
338853,456252,1.00000,1.00000,-1.0,1.00000,0.0,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,1.00000,1.00000,-1.0,1.00000,0.0,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0
338854,456253,1.10000,1.10000,-1.0,1.10000,0.0,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,1.10000,1.10000,-1.0,1.10000,0.0,0.0,-1.0,0.0,0.000000,0.0,-1.0,0.000000,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,-1.0,0.0
338855

In [10]:
# utils.remove_feature(base)

In [11]:
train = utils.get_pickles("train", [KEY])
test = utils.get_pickles("test", [KEY])

In [12]:
_train = pd.merge(train, base, on=KEY, how="left")
_test = pd.merge(test, base, on=KEY, how="left")

In [13]:
PREF = "f106_"

utils.to_feature(_train.drop([KEY], axis=1).add_prefix(PREF), name="train")
utils.to_feature(_test.drop([KEY], axis=1).add_prefix(PREF), name="test")

d:\Data Science/data/feature/train/f106_NAME_CONTRACT_STATUS-Approved.f
d:\Data Science/data/feature/train/f106_NAME_CONTRACT_STATUS-Approved_app.f
d:\Data Science/data/feature/train/f106_NAME_CONTRACT_STATUS-Approved_ref.f
d:\Data Science/data/feature/train/f106_NAME_CONTRACT_STATUS-Approved_appref.f
d:\Data Science/data/feature/train/f106_NAME_CONTRACT_STATUS-Canceled.f
d:\Data Science/data/feature/train/f106_NAME_CONTRACT_STATUS-Canceled_app.f
d:\Data Science/data/feature/train/f106_NAME_CONTRACT_STATUS-Canceled_ref.f
d:\Data Science/data/feature/train/f106_NAME_CONTRACT_STATUS-Canceled_appref.f
d:\Data Science/data/feature/train/f106_NAME_CONTRACT_STATUS-Refused.f
d:\Data Science/data/feature/train/f106_NAME_CONTRACT_STATUS-Refused_app.f
d:\Data Science/data/feature/train/f106_NAME_CONTRACT_STATUS-Refused_ref.f
d:\Data Science/data/feature/train/f106_NAME_CONTRACT_STATUS-Refused_appref.f
d:\Data Science/data/feature/train/f106_NAME_CONTRACT_STATUS-Unused offer.f
d:\Data Science/dat